# 3D segmentation using CellPose

This notebook demonstrates a complete 3D deep learning segmentation workflow using CellPose. It loads a 3D volume with BioIO and Dask for memory efficiency, visualizes it in Napari's interactive viewer, applies GPU-accelerated Cellpose for automated 3D cell segmentation, and overlays the resulting masks on the original volume for evaluation. It requires bioio, napari, and cellpose with GPU support.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Load and visualize 3D microscopy data** using BioIO and Napari
    - Understand how to efficiently handle large 3D images with Dask arrays
    - Navigate and render 3D volumes in Napari's viewer

2. **Apply deep learning segmentation to 3D data** using Cellpose
    - Configure Cellpose for 3D segmentation tasks
    - Understand key parameters: `diameter`, `z_axis`, and `do_3D`
    - Leverage GPU acceleration for faster processing

3. **Evaluate and refine segmentation results**
    - Visualize segmentation masks overlaid on original data
    - Identify when parameters need adjustment (e.g., cell diameter)
    - Recognize scenarios requiring model retraining

4. **Develop practical skills for 3D image analysis workflows**
    - Build reproducible analysis pipelines in Jupyter notebooks
    - Work with real biological imaging data (Lund dataset)
    - Understand the relationship between image properties and segmentation parameters

### Google Colab

If you are using Google Colab to run this, you will need to run the following cell to download the data and some needed packages.

In [1]:
# ============================
# Inicialización estable en Colab
# ============================

# 1. Fijar NumPy en versión estable compatible con bioio
!pip install --upgrade numpy==1.26.4

# 2. Reinstalar bioio contra esa versión de NumPy
!pip install --force-reinstall bioio

# 3. Instalar librerías principales de segmentación y visualización
!pip install cellpose napari napari-colab

# 4. Fijar tifffile en versión estable compatible con bioio-tifffile
!pip install tifffile==2024.5.22

# 5. Instalar bioio-tifffile que usa la versión correcta de tifffile
!pip install --force-reinstall bioio-tifffile

# 6. Fijar zarr en versión <3 (evita el error con ZarrTiffStore)
!pip install "zarr<3"

# 7. Descargar datos de ejemplo
!mkdir -p data
!curl -L https://git.mpi-cbg.de/rhaase/clij2_example_data/-/raw/master/lund1051_resampled.tif --output data/lund1051_resampled.tif


  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
roifile 2026.7.30 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
kerchunk 0.2.10 requires zarr>=3.0.1, but you have zarr 2.18.7 which is incompatible.
tifffile 2026.7.31 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
imagecodecs 2026.6.26 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
xarray-einstats 0.10.0 requires

  Using cached bioio-3.5.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached bioio_base-3.5.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached dask-2026.7.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached ome_types-0.6.3-py3-none-any.whl.metadata (9.8 kB)
  Using cached semver-3.0.4-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached pint-0.25.3-py3-none-any.whl.metadata (10 kB)
  Using cached xarray-2026.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached partd-1.4.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB

  Using cached zarr-3.3.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached zarr-3.3.0-py3-none-any.whl (363 kB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
  Attempting uninstall: zarr
    Found existing installation: zarr 2.18.7
    Uninstalling zarr-2.18.7:
      Successfully uninstalled zarr-2.18.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
pytensor 2.38.3 requires nu

  Using cached tifffile-2024.5.22-py3-none-any.whl.metadata (30 kB)
Using cached tifffile-2024.5.22-py3-none-any.whl (225 kB)
  Attempting uninstall: tifffile
    Found existing installation: tifffile 2026.7.31
    Uninstalling tifffile-2026.7.31:
      Successfully uninstalled tifffile-2026.7.31
  Using cached bioio_tifffile-1.3.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached bioio_base-3.5.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached dask-2026.7.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl.metadata (23 kB)
  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached tifffile-2026.7.31-py3-none-any.whl.metadata (33 kB)
  Using cached xarray-2026.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached ome_types-0.6.3-py3-none-any.whl.metadata (9.8 kB)
  Using cached pint-0.25.3-py3-none-any.

  Using cached zarr-2.18.7-py3-none-any.whl.metadata (5.8 kB)
  Using cached numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.9 kB)
Using cached zarr-2.18.7-py3-none-any.whl (211 kB)
Using cached numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.9 MB)
  Attempting uninstall: numcodecs
    Found existing installation: numcodecs 0.16.5
    Uninstalling numcodecs-0.16.5:
      Successfully uninstalled numcodecs-0.16.5
  Attempting uninstall: zarr
    Found existing installation: zarr 3.3.0
    Uninstalling zarr-3.3.0:
      Successfully uninstalled zarr-3.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kerchunk 0.2.10 requires zarr>=3.0.1, but you have zarr 2.18.7 which is incompatible.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                            

## Load the data

Using [BioIO](https://bioio-devs.github.io/bioio/OVERVIEW.html) with [Dask](https://docs.dask.org/en/stable/) lets us stream the 3D volume lazily so it loads without blowing up memory before sending it to Napari and Cellpose.

In [7]:
from bioio import BioImage

In [8]:
image_handle = BioImage("data/lund1051_resampled.tif")


## Visualize in Napari

We'll use [Napari](https://napari.org/stable/) to interactively visualize the 3D volume.

In [9]:
import napari

# Use Dask for lazy loading of the heavy 3D image
# BioImage's .dask_data provides a Dask array without loading into memory
image_data = image_handle.dask_data.rechunk().squeeze()
image_data

dask.array<getitem, shape=(213, 710, 355), dtype=float32, chunksize=(181, 609, 303), chunktype=numpy.ndarray>

### Google colab

If you are using Google colab, you can open napari in a separate window using noVNC and the following cell. Remember to click on the link that appears at the end.

In [19]:
from napari_colab import setup, open_viewer, screenshot, shutdown
viewer = open_viewer(width=1800, height=1000)

napari already running! returning existing proxy.


### Local run

Otherwise you can open the napari viewer with the cell below.

In [ ]:
#import napari
# Create a Napari viewer and add the image
#viewer = napari.Viewer()


Available platform plugins are: minimal, xcb, vkkhrdisplay, offscreen, wayland-brcm, wayland-egl, wayland, eglfs, linuxfb, vnc, minimalegl.


Available platform plugins are: minimal, xcb, vkkhrdisplay, offscreen, wayland-brcm, wayland-egl, wayland, eglfs, linuxfb, vnc, minimalegl.



### Both

You can see the image with the following cell

In [12]:
viewer.add_image(image_data, name='Image surface')

## Load and run CellPose in GPU

Cellpose is an open-source deep learning model for general cell and nucleus segmentation in microscopy images. In this notebook, it will be used to segment nuclei in the 3D volume. For details, see the [Cellpose documentation](https://cellpose.readthedocs.io/).

In [15]:
from cellpose import io, models, core

# Initialize Cellpose model with GPU enabled
model = models.CellposeModel(gpu=True)

io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

[GUI INFO] : WRITING LOG OUTPUT TO /root/.cellpose/run.log

cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.12.13 
torch version:  	2.11.0+cu128
2026-08-06 15:21:31,119 [io INFO] WRITING LOG OUTPUT TO /root/.cellpose/run.log
2026-08-06 15:21:31,120 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.12.13 
torch version:  	2.11.0+cu128
2026-08-06 15:21:31,122 [core INFO] ** TORCH CUDA version installed and working. **


In [16]:
# Run Cellpose segmentation in 3D mode
masks, flows, styles = model.eval(
    image_data.compute(),
    diameter=100,  # Adjust based on typical cell size in pixels
    z_axis=0,  # Specify the z-axis for 3D data
    do_3D=True  # Enable 3D segmentation
)

2026-08-06 15:21:35,207 [models INFO] resizing 3D image with anisotropy=None
2026-08-06 15:21:35,267 [core INFO] running YX: 63 planes of size (213, 106)
2026-08-06 15:21:56,756 [utils INFO] 100%|##########| 8/8 [00:21<00:00,  2.69s/it]
2026-08-06 15:21:56,772 [core INFO] running ZY: 213 planes of size (63, 106)
2026-08-06 15:23:21,211 [utils INFO] 100%|##########| 27/27 [01:24<00:00,  3.13s/it]
2026-08-06 15:23:21,226 [core INFO] running ZX: 106 planes of size (63, 213)
2026-08-06 15:24:02,686 [utils INFO] 100%|##########| 14/14 [00:41<00:00,  2.96s/it]
2026-08-06 15:24:02,713 [models INFO] resizing 3D flows and cellprob to original image size
2026-08-06 15:24:03,979 [models INFO] network run in 148.77s


/usr/local/lib/python3.12/dist-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


2026-08-06 15:24:08,401 [models INFO] masks created in 4.42s


In [18]:
#viewer = napari.Viewer()
viewer.add_image(image_data, name="Lund volume", rendering="attenuated_mip")
viewer.add_labels(masks, name="Cellpose masks", opacity=0.5)
# viewer.dims.ndisplay = 3
# napari.run()

## Questions

1. Is there any parameter that should be corrected when running CellPose?

2. Do we need to retrain the network?

### Harder questions

1. Can you run Cellpose on the dataset called `lund1051_resampled.tif`? What happens when you choose the right size of the cell nucleus?